In [1]:
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBRegressor
from pathlib import Path


df = pd.read_csv('../data/features_dataset.csv', parse_dates=['date'])

exclude_cols = ['date', 'target', 'target_direction', 'rate_regime']
feature_cols = [c for c in df.columns if c not in exclude_cols and df[c].dtype in ['float64', 'int64', 'int32']]

train = df.dropna(subset=['target'])
X = train[feature_cols]
y = train['target']

model = XGBRegressor(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
model.fit(X, y)

joblib.dump({'model': model, 'features': feature_cols}, '../data/model.joblib')
print(f"Model saved!")
print(f"Features: {len(feature_cols)}")
print(f"Training rows: {len(X)}")

Model saved!
Features: 37
Training rows: 61


In [2]:
# Test the saved model
saved = joblib.load('../data/model.joblib')
model = saved['model']
features = saved['features']

latest = df.iloc[-1]
X_latest = latest[features].values.reshape(1, -1)
prediction = model.predict(X_latest)[0]
current_rate = latest['tbill_91']

print(f"Current rate:    {current_rate:.2f}%")
print(f"Predicted next:  {prediction:.2f}%")
print(f"Change:          {prediction - current_rate:+.2f}%")

Current rate:    10.98%
Predicted next:  11.12%
Change:          +0.14%


In [3]:
api_code = """from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from datetime import datetime

app = FastAPI(
    title="GH-Yield API",
    description="Ghana 91-Day Treasury Bill Rate Forecasting System",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

DATA_PATH = Path(__file__).parent.parent / "data"


def get_prediction():
    saved = joblib.load(DATA_PATH / "model.joblib")
    model = saved["model"]
    features = saved["features"]
    df = pd.read_csv(DATA_PATH / "features_dataset.csv", parse_dates=["date"])
    latest = df.iloc[-1]
    X = latest[features].values.reshape(1, -1)
    predicted = float(model.predict(X)[0])
    current = float(latest["tbill_91"])
    change = predicted - current
    if change < -0.5:
        signal = "BUY"
    elif change > 0.5:
        signal = "WAIT"
    else:
        signal = "HOLD"
    return {
        "date": str(latest["date"]),
        "current_rate": round(current, 2),
        "predicted_next_month": round(predicted, 2),
        "predicted_change": round(change, 2),
        "signal": signal
    }


@app.get("/")
def root():
    return {
        "project": "GH-Yield",
        "description": "Ghana T-Bill Rate Forecasting API",
        "endpoints": ["/predict", "/signal", "/health", "/history"]
    }


@app.get("/predict")
def predict():
    return get_prediction()


@app.get("/signal")
def signal():
    result = get_prediction()
    if result["signal"] == "BUY":
        result["recommendation"] = f"Lock in the current {result['current_rate']}% rate. Model predicts decline."
    elif result["signal"] == "WAIT":
        result["recommendation"] = f"Hold cash. Model predicts rates will rise to {result['predicted_next_month']}%."
    else:
        result["recommendation"] = "Maintain current position. No significant change expected."
    result["timestamp"] = datetime.now().isoformat()
    return result


@app.get("/health")
def health():
    try:
        saved = joblib.load(DATA_PATH / "model.joblib")
        return {
            "status": "healthy",
            "model_loaded": True,
            "n_features": len(saved["features"]),
            "timestamp": datetime.now().isoformat()
        }
    except Exception as e:
        return {"status": "unhealthy", "error": str(e)}


@app.get("/history")
def history():
    df = pd.read_csv(DATA_PATH / "ghana_tbill_rates.csv", parse_dates=["date"])
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df.to_dict(orient="records")
"""

with open('../src/api.py', 'w') as f:
    f.write(api_code)

print("Created: src/api.py")

Created: src/api.py


In [4]:
dashboard_code = """import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import joblib
from pathlib import Path

st.set_page_config(page_title="GH-Yield Dashboard", page_icon="", layout="wide")
st.title("GH-Yield: Ghana T-Bill Forecasting System")
st.markdown("Intelligent Treasury Bill Rate Forecasting for Ghana")

DATA_PATH = Path(__file__).parent.parent / "data"


@st.cache_data
def load_data():
    tbill = pd.read_csv(DATA_PATH / "ghana_tbill_rates.csv", parse_dates=["date"])
    features = pd.read_csv(DATA_PATH / "features_dataset.csv", parse_dates=["date"])
    try:
        signals = pd.read_csv(DATA_PATH / "predictions_with_signals.csv", parse_dates=["date"])
    except:
        signals = None
    try:
        strategy = pd.read_csv(DATA_PATH / "backtest_strategy.csv", parse_dates=["date"])
        benchmark = pd.read_csv(DATA_PATH / "backtest_benchmark.csv", parse_dates=["date"])
    except:
        strategy = None
        benchmark = None
    return tbill, features, signals, strategy, benchmark


def get_prediction():
    saved = joblib.load(DATA_PATH / "model.joblib")
    model = saved["model"]
    feature_cols = saved["features"]
    df = pd.read_csv(DATA_PATH / "features_dataset.csv", parse_dates=["date"])
    latest = df.iloc[-1]
    X = latest[feature_cols].values.reshape(1, -1)
    predicted = float(model.predict(X)[0])
    current = float(latest["tbill_91"])
    change = predicted - current
    if change < -0.5:
        signal = "BUY"
    elif change > 0.5:
        signal = "WAIT"
    else:
        signal = "HOLD"
    return {
        "current_rate": round(current, 2),
        "predicted_next_month": round(predicted, 2),
        "predicted_change": round(change, 2),
        "signal": signal
    }


tbill, features, signals, strategy, benchmark = load_data()
prediction = get_prediction()

# KPI Cards
st.header("Current Status")
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(label="91-Day Rate", value=f"{prediction['current_rate']:.2f}%")

with col2:
    st.metric(
        label="Predicted Next Month",
        value=f"{prediction['predicted_next_month']:.2f}%",
        delta=f"{prediction['predicted_change']:+.2f}%"
    )

with col3:
    signal_icon = {"BUY": "GREEN", "WAIT": "RED", "HOLD": "YELLOW"}
    st.metric(label="Signal", value=f"{prediction['signal']}")

with col4:
    slope = tbill["tbill_364"].iloc[-1] - tbill["tbill_91"].iloc[-1]
    st.metric(label="Yield Curve Slope", value=f"{slope:.2f}%")

# Rate History
st.header("Rate History")
fig = go.Figure()
fig.add_trace(go.Scatter(x=tbill["date"], y=tbill["tbill_91"], name="91-Day", line=dict(width=2)))
fig.add_trace(go.Scatter(x=tbill["date"], y=tbill["tbill_182"], name="182-Day", line=dict(width=2, dash="dash")))
fig.add_trace(go.Scatter(x=tbill["date"], y=tbill["tbill_364"], name="364-Day", line=dict(width=2, dash="dot")))
fig.add_trace(go.Scatter(x=tbill["date"], y=tbill["policy_rate"], name="Policy Rate", line=dict(width=1, color="red")))
fig.update_layout(yaxis_title="Rate (%)", hovermode="x unified", height=450)
st.plotly_chart(fig, use_container_width=True)

# Yield Curve
st.header("Current Yield Curve")
latest_row = tbill.iloc[-1]
tenors = ["91-Day", "182-Day", "364-Day"]
rates = [latest_row["tbill_91"], latest_row["tbill_182"], latest_row["tbill_364"]]
fig_yc = go.Figure()
fig_yc.add_trace(go.Scatter(x=tenors, y=rates, mode="lines+markers", line=dict(width=3, color="blue"), marker=dict(size=10)))
fig_yc.update_layout(yaxis_title="Rate (%)", xaxis_title="Tenor", height=350)
st.plotly_chart(fig_yc, use_container_width=True)

# Trading Signals
if signals is not None:
    st.header("Trading Signals History")
    fig_sig = go.Figure()
    fig_sig.add_trace(go.Scatter(x=signals["date"], y=signals["current_rate"], name="Actual Rate", line=dict(width=2, color="blue")))
    fig_sig.add_trace(go.Scatter(x=signals["date"], y=signals["predicted_next"], name="Predicted", line=dict(width=1, dash="dash", color="red")))
    for _, row in signals.iterrows():
        color = {"BUY": "green", "WAIT": "red", "HOLD": "gray"}[row["signal"]]
        fig_sig.add_vrect(x0=row["date"], x1=row["date"] + pd.Timedelta(days=28), fillcolor=color, opacity=0.1, line_width=0)
    fig_sig.update_layout(yaxis_title="Rate (%)", height=400)
    st.plotly_chart(fig_sig, use_container_width=True)

# Backtest
if strategy is not None and benchmark is not None:
    st.header("Backtest Performance")
    col1, col2 = st.columns(2)
    with col1:
        ret = (strategy["value"].iloc[-1] / 1_000_000 - 1) * 100
        st.metric("Strategy Return", f"{ret:.2f}%")
    with col2:
        ret_bm = (benchmark["value"].iloc[-1] / 1_000_000 - 1) * 100
        st.metric("Benchmark Return", f"{ret_bm:.2f}%")
    fig_bt = go.Figure()
    fig_bt.add_trace(go.Scatter(x=strategy["date"], y=strategy["value"], name="ML Strategy", line=dict(width=2, color="blue")))
    fig_bt.add_trace(go.Scatter(x=benchmark["date"], y=benchmark["value"], name="Buy & Hold", line=dict(width=2, dash="dash", color="red")))
    fig_bt.update_layout(yaxis_title="Portfolio Value (GHS)", height=400)
    st.plotly_chart(fig_bt, use_container_width=True)

# Data Table
st.header("Raw Data")
st.dataframe(tbill.sort_values("date", ascending=False), use_container_width=True)

# Footer
st.markdown("---")
st.markdown("**GH-Yield** | Built by David Quayefio | [GitHub](https://github.com/david006-DS/GH-Yield)")
"""

with open('../dashboards/app.py', 'w') as f:
    f.write(dashboard_code)

print("Created: dashboards/app.py")

Created: dashboards/app.py


In [5]:
dockerfile = """FROM python:3.12-slim

WORKDIR /app

COPY --from=ghcr.io/astral-sh/uv:latest /uv /bin/uv

COPY pyproject.toml uv.lock ./
COPY src/ src/
COPY data/ data/
COPY dashboards/ dashboards/

RUN uv sync --frozen --no-dev

EXPOSE 8000 8501

CMD ["uv", "run", "uvicorn", "src.api:app", "--host", "0.0.0.0", "--port", "8000"]
"""

docker_compose = """version: "3.8"

services:
  api:
    build: .
    ports:
      - "8000:8000"
    command: uv run uvicorn src.api:app --host 0.0.0.0 --port 8000

  dashboard:
    build: .
    ports:
      - "8501:8501"
    command: uv run streamlit run dashboards/app.py --server.port 8501 --server.address 0.0.0.0
"""

with open('../Dockerfile', 'w') as f:
    f.write(dockerfile)

with open('../docker-compose.yml', 'w') as f:
    f.write(docker_compose)

print("Created: Dockerfile")
print("Created: docker-compose.yml")

Created: Dockerfile
Created: docker-compose.yml


In [6]:
print("""
============================================================
DEPLOYMENT FILES CREATED
============================================================

Files generated:
  - data/model.joblib          (trained model)
  - src/api.py                 (FastAPI REST API)
  - dashboards/app.py          (Streamlit dashboard)
  - Dockerfile                 (container config)
  - docker-compose.yml         (multi-container setup)

============================================================
HOW TO TEST LOCALLY
============================================================

1. Test the API (open a new Git Bash terminal):

   cd ~/Desktop/gh-yield/src
   uv run uvicorn api:app --reload --port 8000

   Then open in browser:
   - http://localhost:8000          (API root)
   - http://localhost:8000/predict  (forecast)
   - http://localhost:8000/signal   (trading signal)
   - http://localhost:8000/health   (health check)
   - http://localhost:8000/history  (historical data)

   Press Ctrl+C to stop.

2. Test the dashboard (open another Git Bash terminal):

   cd ~/Desktop/gh-yield
   uv run streamlit run dashboards/app.py

   Opens at http://localhost:8501

3. Commit everything:

   cd ~/Desktop/gh-yield
   git add .
   git commit -m "Week 4: FastAPI, Streamlit dashboard, Docker deployment"
   git push

============================================================
""")


DEPLOYMENT FILES CREATED

Files generated:
  - data/model.joblib          (trained model)
  - src/api.py                 (FastAPI REST API)
  - dashboards/app.py          (Streamlit dashboard)
  - Dockerfile                 (container config)
  - docker-compose.yml         (multi-container setup)

HOW TO TEST LOCALLY

1. Test the API (open a new Git Bash terminal):

   cd ~/Desktop/gh-yield/src
   uv run uvicorn api:app --reload --port 8000

   Then open in browser:
   - http://localhost:8000          (API root)
   - http://localhost:8000/predict  (forecast)
   - http://localhost:8000/signal   (trading signal)
   - http://localhost:8000/health   (health check)
   - http://localhost:8000/history  (historical data)

   Press Ctrl+C to stop.

2. Test the dashboard (open another Git Bash terminal):

   cd ~/Desktop/gh-yield
   uv run streamlit run dashboards/app.py

   Opens at http://localhost:8501

3. Commit everything:

   cd ~/Desktop/gh-yield
   git add .
   git commit -m "Week 4: Fa